---

# Implementación en reglas del proyecto <a class="anchor" id="comp"></a>


([Volver al inicio](#inicio))

Usamos los tres recorridos de tableaux de este taller (`primero_anchura`, `primero_profundidad` y `backtracking`) con las reglas de nuestro proyecto (`problema.py`): un grafo de 5 ciudades y 7 carreteras, con el descriptor $Visita(c,t)$ = «la ciudad $c$ se visita en la posición $t$» (25 letras proposicionales).

Corremos **cada regla por separado** y luego **las cinco juntas**. Con las reglas completas el tableaux ramifica muchísimo (cada `O` y cada `>` abre dos ramas), así que cada recorrido tiene un **límite de tiempo**: si lo agota, lo registramos como «tiempo agotado».

In [18]:
import os, sys, time, signal
import pandas as pd
from logica import *
from problema import CicloHamiltoniano

sys.setrecursionlimit(10000)   # las reglas son fórmulas largas (recursión profunda)
os.makedirs('resultados', exist_ok=True)

ciudades = ['Baq', 'Buc', 'Bog', 'Med', 'Per']
aristas  = [{0,1}, {1,2}, {2,3}, {3,4}, {2,4}, {0,2}, {1,4}]
coords   = {0:(0,3), 1:(3,3), 2:(1.5,1.8), 3:(0.3,0), 4:(2.7,0)}   # solo para dibujar

H = CicloHamiltoniano(ciudades, aristas, coords)
print('Letras proposicionales:', H.n * H.n)

# Compatibilidad entre versiones de Logica: algunas versiones de `expandir` devuelven
# hijos None (por ejemplo [nodo, None]). Los descartamos para que los tres recorridos
# funcionen igual con cualquiera de ellas.
if not getattr(nodos_tableaux, '_sin_none', False):
    _expandir_original = nodos_tableaux.expandir
    def _expandir_sin_none(self):
        return [h for h in _expandir_original(self) if h is not None]
    nodos_tableaux.expandir = _expandir_sin_none
    nodos_tableaux._sin_none = True

Letras proposicionales: 25


## Límite de tiempo y registro de resultados

* `MAX_SEG`: segundos que se le dan a **cada** recorrido en **cada** fórmula. Súbelo si quieres esperar más.
* El límite usa `signal.alarm`, que funciona en Mac y Linux (no en Windows).
* `correr_tableaux` corre los tres recorridos sobre una fórmula y guarda una fila por recorrido en `resultados`; la tabla resumen sale de ahí. Si repites una celda, sus filas se actualizan (no se duplican).
* En las filas con «tiempo agotado», el tiempo mostrado es el límite, no un tiempo de solución.

In [19]:
import time, signal
import pandas as pd
from logica import inorder_to_tree, nodos_tableaux, Ytoria

MAX_SEG = 3600        # segundos que se le dan a CADA recorrido en CADA fórmula

class TiempoAgotado(Exception):
    pass

def _alarma(signum, frame):
    raise TiempoAgotado()

signal.signal(signal.SIGALRM, _alarma)      # signal.alarm: Mac y Linux (no Windows)

resultados = {}      # (solver, regla) -> fila
modelos_A = {}       # modelos de las 5 juntas (para visualizar)

def correr_tableaux(nombre, formula, max_seg=None):
    '''Corre los tres recorridos (métodos de Formula), cada uno con un límite de tiempo.'''
    limite = MAX_SEG if max_seg is None else max_seg
    for etiqueta, metodo in [('Tableaux anchura',      'SATtableaux_anchura'),
                             ('Tableaux profundidad',  'SATtableaux_profundidad'),
                             ('Tableaux backtracking', 'SATtableaux_backtracking')]:
        A = inorder_to_tree(formula)          # instancia nueva por recorrido
        nodo = nodos_tableaux([A])
        funcion = getattr(A, metodo)          # método ya ligado a A
        I, estado = None, 'sin modelo'
        t0 = time.perf_counter()
        signal.alarm(limite)
        try:
            I = funcion(nodo)
            estado = 'modelo' if I else 'sin modelo'
        except TiempoAgotado:
            estado = 'tiempo agotado'
        except RecursionError:
            estado = 'RecursionError'
        finally:
            signal.alarm(0)
        seg = time.perf_counter() - t0
        verifica = None
        if I:
            J = {l: False for l in A.letras()}
            J.update(I)
            verifica = A.valor(J)
            if nombre == 'Reglas 1 a 5':
                modelos_A[etiqueta] = I
        resultados[(etiqueta, nombre)] = {
            'Solver': etiqueta,
            'Regla': nombre,
            'Resultado': estado,
            '¿Verifica?': verifica,
            'Tiempo (s)': round(seg, 3),
            'Modelo': I,
        }
        print(f'{nombre:14s} | {etiqueta:22s} | {estado:14s} | {seg:7.3f} s')

In [20]:
import time

for k in range(5):
    t0 = time.perf_counter()
    S = H.reglas[k]
    print(f'Regla {k+1}: fórmula de {len(S)} caracteres, {inorder_to_tree(S).num_conec()} conectivos')

Regla 1: fórmula de 97 caracteres, 24 conectivos
Regla 2: fórmula de 97 caracteres, 24 conectivos
Regla 3: fórmula de 522 caracteres, 149 conectivos
Regla 4: fórmula de 301 caracteres, 75 conectivos
Regla 5: fórmula de 73 caracteres, 18 conectivos


## Regla por regla

1. En cada posición hay al menos una ciudad.
2. Cada ciudad está en al menos una posición.
3. Cada ciudad está en a lo sumo una posición.
4. Transiciones válidas: posiciones consecutivas son ciudades conectadas por una carretera.
5. Cierre del ciclo: la última posición conecta con la primera.

**Regla 1**

In [21]:
%%time
correr_tableaux('Regla 1', H.reglas[0])

Regla 1        | Tableaux anchura       | modelo         |   0.018 s
Regla 1        | Tableaux profundidad   | modelo         |   0.003 s
Regla 1        | Tableaux backtracking  | modelo         |   0.009 s
CPU times: user 31 ms, sys: 2.67 ms, total: 33.7 ms
Wall time: 32.2 ms


**Regla 2**


In [22]:
%%time
correr_tableaux('Regla 2', H.reglas[1])

Regla 2        | Tableaux anchura       | modelo         |   0.018 s
Regla 2        | Tableaux profundidad   | modelo         |   0.005 s
Regla 2        | Tableaux backtracking  | modelo         |   0.009 s
CPU times: user 32.5 ms, sys: 2.52 ms, total: 35 ms
Wall time: 33.2 ms


**Regla 3**


In [23]:
%%time
correr_tableaux('Regla 3', H.reglas[2])

Regla 3        | Tableaux anchura       | modelo         |   5.182 s
Regla 3        | Tableaux profundidad   | modelo         |   0.067 s
Regla 3        | Tableaux backtracking  | modelo         |   0.038 s
CPU times: user 5.23 s, sys: 56.9 ms, total: 5.28 s
Wall time: 5.29 s


**Regla 4**


In [24]:
%%time
correr_tableaux('Regla 4', H.reglas[3])

Regla 4        | Tableaux anchura       | modelo         | 438.691 s
Regla 4        | Tableaux profundidad   | modelo         |   0.027 s
Regla 4        | Tableaux backtracking  | modelo         |   0.018 s
CPU times: user 4min 30s, sys: 1min 43s, total: 6min 13s
Wall time: 7min 18s


**Regla 5**


In [25]:
%%time
correr_tableaux('Regla 5', H.reglas[4])

Regla 5        | Tableaux anchura       | modelo         |   0.003 s
Regla 5        | Tableaux profundidad   | modelo         |   0.002 s
Regla 5        | Tableaux backtracking  | modelo         |   0.001 s
CPU times: user 5.65 ms, sys: 411 µs, total: 6.06 ms
Wall time: 5.92 ms


## Las cinco reglas juntas

Es el problema completo. Cada recorrido puede usar todo el `MAX_SEG`, así que esta celda puede tardar hasta `3 * MAX_SEG` segundos. Mientras corre, el kernel está ocupado y no se puede ver la tabla; si no quieres esperar, interrúmpelo.

In [26]:
%%time
correr_tableaux('Reglas 1 a 5', Ytoria(H.reglas))

Reglas 1 a 5   | Tableaux anchura       | tiempo agotado | 3858.243 s
Reglas 1 a 5   | Tableaux profundidad   | tiempo agotado | 3600.003 s
Reglas 1 a 5   | Tableaux backtracking  | tiempo agotado | 3600.010 s
CPU times: user 2h 22min 28s, sys: 16min 28s, total: 2h 38min 57s
Wall time: 3h 23min 54s


### Visualización de la solución

Si algún recorrido encontró un modelo de las cinco reglas, lo dibujamos con el visualizador por turnos (si ninguno lo encontró dentro del límite, no hay nada que dibujar).

In [28]:
for solver, I in modelos_A.items():
    if None not in H._orden(I):       # el modelo asigna una ciudad a cada turno
        print('Modelo encontrado por:', solver)
        H.visualizar_por_turnos(I)
        break
else:
    print('Ningún recorrido encontró un ciclo dentro del límite de', MAX_SEG, 's')

Ningún recorrido encontró un ciclo dentro del límite de 3600 s


## Tabla resumen

Reúne todas las filas registradas arriba (una por recorrido y por regla) y las guarda en un CSV para juntarlas después con las de los otros SAT-solvers.

In [29]:
tabla = pd.DataFrame([{k: v for k, v in fila.items() if k != 'Modelo'}      # el modelo va al JSON, no al CSV
                      for fila in resultados.values()])
tabla.to_csv('resultados/resultados_Tableaux.csv', index=False)
tabla

,Solver,Regla,Resultado,¿Verifica?,Tiempo (s)
0,Tableaux anchura,Regla 1,modelo,True,0.018
1,Tableaux profundidad,Regla 1,modelo,True,0.003
2,Tableaux backtracking,Regla 1,modelo,True,0.009
3,Tableaux anchura,Regla 2,modelo,True,0.018
4,Tableaux profundidad,Regla 2,modelo,True,0.005
5,Tableaux backtracking,Regla 2,modelo,True,0.009
6,Tableaux anchura,Regla 3,modelo,True,5.182
7,Tableaux profundidad,Regla 3,modelo,True,0.067
8,Tableaux backtracking,Regla 3,modelo,True,0.038
9,Tableaux anchura,Regla 4,modelo,True,438.691


## Lectura de los resultados

Completa esta sección con los números de tu corrida. Puntos para comentar:

* **Reglas sueltas:** casi todas son fáciles de satisfacer, así que los tres recorridos las resuelven rápido. La diferencia se nota en las reglas 3 y 4, donde las implicaciones y las disyunciones largas generan muchas ramas.
* **El recorrido importa:** sobre la misma fórmula, anchura, profundidad y backtracking pueden diferir por órdenes de magnitud, porque cada uno elige distinto qué rama explorar primero.
* **Reglas 1 a 5 juntas:** el árbol del tableaux crece de forma exponencial con el número de `O` y `>`, y se llega al límite de tiempo sin encontrar modelo (o con tiempos muy altos). Es el contraste con DPLL sobre la forma clausal de Tseitin.

In [30]:
import json

modelos_Tableaux = {'Tableaux': {}}
for (solver, regla), fila in resultados.items():
    modelos_Tableaux['Tableaux'].setdefault(regla, {})[solver] = {
        'resultado': fila['Resultado'],
        'verifica':  fila['¿Verifica?'],
        'tiempo_s':  fila['Tiempo (s)'],
        'modelo':    fila['Modelo'],
    }

with open('resultados/modelos_Tableaux.json', 'w', encoding='utf-8') as f:
    json.dump(modelos_Tableaux, f, ensure_ascii=False, indent=2)

print('Guardado resultados/modelos_Tableaux.json')
for regla, solvers in modelos_Tableaux['Tableaux'].items():
    con_modelo = [s for s, d in solvers.items() if d['modelo'] is not None]
    print(f'  {regla:14s} → con modelo: {con_modelo if con_modelo else "ninguno"}')

Guardado resultados/modelos_Tableaux.json
  Regla 1        → con modelo: ['Tableaux anchura', 'Tableaux profundidad', 'Tableaux backtracking']
  Regla 2        → con modelo: ['Tableaux anchura', 'Tableaux profundidad', 'Tableaux backtracking']
  Regla 3        → con modelo: ['Tableaux anchura', 'Tableaux profundidad', 'Tableaux backtracking']
  Regla 4        → con modelo: ['Tableaux anchura', 'Tableaux profundidad', 'Tableaux backtracking']
  Regla 5        → con modelo: ['Tableaux anchura', 'Tableaux profundidad', 'Tableaux backtracking']
  Reglas 1 a 5   → con modelo: ninguno
